# Split single-cell data for TGF-β versus vehicle model

We split the feature selected single-cell profiles into data splits for one binary model that predicts TGF-β treated (fibrotic) versus vehicle (non-failing) treated cardiac fibroblasts.

1. **Holdout:** Hold out one whole biological replicate (randomly selected) and one random well per treatment from each of the remaining biological replicates.
2. **Training and testing:** Split the remaining data using 70/30 splitting, stratified by treatment.

In [1]:
import pathlib

import pandas as pd
from sklearn.model_selection import train_test_split

## Set paths and variables

In [2]:
# Set random state for the whole notebook to ensure reproducibility
random_state = 0

# Plate to process
plate = "CARD-CelIns-CX7_261022150001"

# Metadata column used for prediction class and column with biological replicate
label = "Metadata_Treatment"
replicate_col = "Metadata_N_Replicate"

# Path to feature selected single-cell profiles (QC applied)
feature_selected_path = pathlib.Path(
    f"../2.preprocessing_features/data/single_cell_profiles/{plate}_sc_feature_selected.parquet"
).resolve(strict=True)

# Make directory for split data for this model
model_output_dir = pathlib.Path("./data_splits/model_TGFB_vs_Veh")
model_output_dir.mkdir(parents=True, exist_ok=True)

print(f"Found feature selected file: {feature_selected_path.name}")

Found feature selected file: CARD-CelIns-CX7_261022150001_sc_feature_selected.parquet


## Load in feature selected data

In [3]:
# Load the feature selected file as a DataFrame
feature_selected_df = pd.read_parquet(feature_selected_path)

# Drop Location features (position of the cell in the image), which are not biologically meaningful
location_features = [
    col
    for col in feature_selected_df.columns
    if "_Location_" in col and not col.startswith("Metadata_")
]
feature_selected_df = feature_selected_df.drop(columns=location_features)
print(f"Dropped {len(location_features)} Location features: {location_features}")

print(f"Shape: {feature_selected_df.shape}")
print(
    feature_selected_df.groupby([replicate_col, label])
    .agg(n_cells=("Metadata_Well", "size"), n_wells=("Metadata_Well", "nunique"))
    .to_string()
)
feature_selected_df.head()

Dropped 2 Location features: ['Cytoplasm_Location_MaxIntensity_Y_Actin', 'Cells_Location_MaxIntensity_X_Actin']
Shape: (11623, 1051)
                                         n_cells  n_wells
Metadata_N_Replicate Metadata_Treatment                  
N1                   TGFB                   1477        6
                     Veh                    2084        6
N2                   TGFB                   1287        6
                     Veh                    1250        6
N3                   TGFB                   1097        6
                     Veh                    1501        6
N4                   TGFB                   1485        6
                     Veh                    1442        6


,Metadata_WellRow,Metadata_WellCol,Metadata_Treatment,Metadata_N_Replicate,Metadata_Nuclei_Location_Center_X,Metadata_Nuclei_Location_Center_Y,Metadata_Cells_Location_Center_X,Metadata_Cells_Location_Center_Y,Metadata_Image_Count_Cells,Metadata_ImageNumber,...,Nuclei_Texture_InverseDifferenceMoment_ER_3_02_256,Nuclei_Texture_InverseDifferenceMoment_Golgi_3_00_256,Nuclei_Texture_InverseDifferenceMoment_Mito_3_00_256,Nuclei_Texture_SumEntropy_Golgi_3_01_256,Nuclei_Texture_SumEntropy_Mito_3_01_256,Nuclei_Texture_SumVariance_Actin_3_01_256,Nuclei_Texture_SumVariance_DNA_3_01_256,Nuclei_Texture_SumVariance_ER_3_03_256,Nuclei_Texture_SumVariance_Golgi_3_01_256,Nuclei_Texture_Variance_Mito_3_02_256
0,B,3,TGFB,N2,436.220491,216.531456,447.810243,224.585940,8,2,...,0.812879,0.830092,-0.221541,-0.257177,0.423440,2.156955,0.965879,0.000589,-0.265292,1.245460
1,B,3,TGFB,N2,369.889953,727.496963,393.137873,745.939119,4,17,...,-0.856472,-0.285865,-0.683333,1.054113,0.824451,1.146963,3.218424,3.400421,2.378942,2.043076
2,B,3,TGFB,N2,559.629747,124.011076,557.110938,122.258248,29,23,...,0.341242,0.094002,-0.966073,-0.324948,0.735658,3.593057,-0.187015,-0.109037,-0.358719,0.887065
3,B,3,TGFB,N2,357.805947,83.312989,256.710809,121.585914,8,20,...,-3.504845,-1.577004,-1.868181,0.640791,1.004835,3.007694,5.382027,6.562487,1.659085,3.456054
4,B,3,TGFB,N2,918.220513,208.039744,896.895539,222.789267,11,14,...,0.147965,0.591231,-0.113318,-0.808940,-0.326096,0.849336,1.679580,-0.542291,-0.592349,-0.340792


## Perform splits for TGF-β versus vehicle model

Holdout one whole biological replicate (randomly selected) and one random well per treatment from each of the remaining biological replicates.

In [4]:
# Randomly select one biological replicate to hold out (make sampling reproducible)
holdout_replicate = (
    pd.Series(sorted(feature_selected_df[replicate_col].dropna().unique()))
    .sample(1, random_state=random_state)
    .item()
)
print(f"Randomly selected biological replicate for holdout: {holdout_replicate}")

# Hold out all rows from the selected biological replicate
holdout_mask = feature_selected_df[replicate_col] == holdout_replicate

# Randomly hold out one well per treatment per other biological replicate (make sampling reproducible)
random_wells = (
    feature_selected_df[~holdout_mask]
    .groupby([replicate_col, label])["Metadata_Well"]
    .apply(lambda x: x.dropna().sample(1, random_state=random_state))
    .explode()
)
print(f"Randomly selected wells for holdout: {random_wells.tolist()}")

# Combine holdout replicate rows with random well rows
well_mask = feature_selected_df["Metadata_Well"].isin(random_wells)
holdout_idx = feature_selected_df.index[holdout_mask | well_mask]

# Create holdout and remaining dataframes
holdout_df = feature_selected_df.loc[holdout_idx].copy()
model_df = feature_selected_df.drop(holdout_idx).copy()

# Record which holdout type each cell belongs to (whole replicate or random well)
holdout_df["Metadata_holdout_type"] = "well"
holdout_df.loc[
    holdout_df[replicate_col] == holdout_replicate, "Metadata_holdout_type"
] = "replicate"

# Save holdout set
holdout_df.to_parquet(model_output_dir / "holdout_split.parquet")

print(f"Holdout set shape: {holdout_df.shape}")
print(
    holdout_df.groupby(["Metadata_holdout_type", replicate_col, label])["Metadata_Well"]
    .agg(n_cells="size", wells=lambda s: sorted(s.unique()))
    .to_string()
)

Randomly selected biological replicate for holdout: N3
Randomly selected wells for holdout: ['B07', 'G04', 'D10', 'F05', 'B06', 'F03']


Holdout set shape: (4269, 1052)
                                                               n_cells                           wells
Metadata_holdout_type Metadata_N_Replicate Metadata_Treatment                                         
replicate             N3                   TGFB                   1097  [C09, D06, E03, E06, F07, G09]
                                           Veh                    1501  [B04, C06, C07, D05, F04, F10]
well                  N1                   TGFB                    327                           [B07]
                                           Veh                     375                           [G04]
                      N2                   TGFB                    203                           [D10]
                                           Veh                     260                           [F05]
                      N4                   TGFB                    300                           [B06]
                                         

In [5]:
print(f"Model data shape (after dropping holdout rows): {model_df.shape}")

# Sanity checks
assert (
    holdout_df.shape[0] + model_df.shape[0] == feature_selected_df.shape[0]
), "Holdout + remaining does not equal original after splitting"
assert (
    holdout_replicate not in model_df[replicate_col].unique()
), "Holdout replicate found in the remaining data"

# Set the ratio of the test data to 30% (training data will be 70%)
test_ratio = 0.30

# Split data into training and test sets
train_df, test_df = train_test_split(
    model_df,
    test_size=test_ratio,
    stratify=model_df[label],
    random_state=random_state,
)

# Save training and test data
train_df.to_parquet(model_output_dir / "training_split.parquet")
test_df.to_parquet(model_output_dir / "testing_split.parquet")

print(f"Training data shape: {train_df.shape}")
print(f"Testing data shape: {test_df.shape}")

# Print out the proportion of samples per treatment in each split
print("\nTreatment distribution in training set:")
print(train_df[label].value_counts(normalize=True))
print("\nTreatment distribution in testing set:")
print(test_df[label].value_counts(normalize=True))

Model data shape (after dropping holdout rows): (7354, 1051)


Training data shape: (5147, 1051)
Testing data shape: (2207, 1051)

Treatment distribution in training set:
Metadata_Treatment
Veh     0.535069
TGFB    0.464931
Name: proportion, dtype: float64

Treatment distribution in testing set:
Metadata_Treatment
Veh     0.535116
TGFB    0.464884
Name: proportion, dtype: float64
